# Ìmọ̀lára · `e2g_bilstm_word2vec_reg_final`

E2g: E2d with stronger regularisation (all E2 runs peak at epoch 2-3, then overfit)

Runs `python -m src.rnn --config configs/e2g_bilstm_word2vec_reg.yaml` for seeds 42, 43, 44 with `--final` on a Kaggle T4 GPU.

* **Code:** [Hassan-Adelani-Luqman/imolara-yoruba-sentiment](https://github.com/Hassan-Adelani-Luqman/imolara-yoruba-sentiment) at commit [`56c3ecd`](https://github.com/Hassan-Adelani-Luqman/imolara-yoruba-sentiment/tree/56c3ecd62fa6c61f970c537d58f9a5ac66902574), cloned below; nothing is edited inside this notebook.
* **Data:** AfriSenti Yoruba (raw TSVs, pinned commit), downloaded by `src/data.py`.
* **Outputs:** `/kaggle/working/outputs/seed*/` (metrics, dev predictions, run info), downloaded to `results/kaggle/e2g_bilstm_word2vec_reg_final/` in the repo.
* **Reproduce:** *Copy & Edit*, set Accelerator = GPU T4 and Internet = on and attach the `imolara-embeddings` dataset, then *Run all*.

Generated by `scripts/kaggle_run.py --notebook`.

## 1 · Setup: clone the repo at the pinned commit, install extras, check the GPU

In [1]:
import glob, json, os, subprocess, sys

REPO, COMMIT = "https://github.com/Hassan-Adelani-Luqman/imolara-yoruba-sentiment.git", "56c3ecd62fa6c61f970c537d58f9a5ac66902574"
CODE, OUT = "/tmp/imolara", "/kaggle/working/outputs"
CONFIG, SEEDS = "configs/e2g_bilstm_word2vec_reg.yaml", [42, 43, 44]

subprocess.run(["git", "clone", "-q", REPO, CODE], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements-kaggle.txt"], check=True)
# Kaggle ships torchao 0.10, which peft >= 0.20 rejects at import time; we don't use it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)

import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU T4"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/env.txt", "w") as f:            # exact package versions, for the report
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f, check=True)

env = {**os.environ, "IMOLARA_GIT_COMMIT": COMMIT[:7], "PYTHONUNBUFFERED": "1"}
emb = glob.glob("/kaggle/input/**/w2v_yo_300.kv", recursive=True)
if emb:                                            # attached imolara-embeddings dataset (RNN runs)
    env["IMOLARA_EMB_DIR"] = os.path.dirname(emb[0])
    print("embeddings:", env["IMOLARA_EMB_DIR"])

GPU: Tesla T4 | torch 2.11.0+cu128
embeddings: /kaggle/input/datasets/luqmanhassanadelani/imolara-embeddings


## 2 · Experiment configuration

In [2]:
print(open(f"{CODE}/{CONFIG}").read())

# E2g: E2d with stronger regularisation (all E2 runs peak at epoch 2-3, then overfit)
embeddings: word2vec
dropout: 0.5
hidden: 64
weight_decay: 1.0e-5



## 3 · Training: one process per seed, one GPU per process

Kaggle's T4 machine has two GPUs. Each seed trains in its own process pinned to one GPU (`CUDA_VISIBLE_DEVICES`), so two seeds run at once and `batch_size` in the config is the real batch (no DataParallel). Log lines are prefixed with their seed. The best epoch is chosen on *clean* dev macro-F1 (dev tweets that do not duplicate a training tweet).

In [3]:
import queue, threading

N_GPU = max(1, torch.cuda.device_count())
print(f"{N_GPU} GPU(s) -> up to {N_GPU} seeds in parallel", flush=True)
NOISE = ("Warning", "warn(", "it/s]", "Loading weights")

def run(seed, gpu):
    cmd = [sys.executable, "-m", "src.rnn", "--config", CONFIG, "--seed", str(seed),
           "--output_dir", f"{OUT}/seed{seed}", *['--final']]
    proc = subprocess.Popen(cmd, cwd=CODE, env={**env, "CUDA_VISIBLE_DEVICES": str(gpu)},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if not any(n in line for n in NOISE):
            print(f"[seed {seed} | gpu {gpu}] {line}", end="", flush=True)
    return proc.wait()

def run_all(seeds):
    todo, codes = queue.Queue(), {}
    for s in seeds:
        todo.put(s)
    def worker(gpu):
        while True:
            try:
                seed = todo.get_nowait()
            except queue.Empty:
                return
            codes[seed] = run(seed, gpu)
    threads = [threading.Thread(target=worker, args=(g,)) for g in range(N_GPU)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    failed = [s for s, rc in codes.items() if rc != 0]
    assert not failed, f"seeds failed: {failed}"

run_all(SEEDS)

2 GPU(s) -> up to 2 seeds in parallel
[seed 42 | gpu 0] epoch  1  loss 0.9088  clean-dev macro-F1 0.6253
[seed 43 | gpu 1] epoch  1  loss 0.9040  clean-dev macro-F1 0.5396
[seed 42 | gpu 0] epoch  2  loss 0.7033  clean-dev macro-F1 0.6360
[seed 43 | gpu 1] epoch  2  loss 0.7086  clean-dev macro-F1 0.6657
[seed 42 | gpu 0] epoch  3  loss 0.5468  clean-dev macro-F1 0.6858
[seed 43 | gpu 1] epoch  3  loss 0.5436  clean-dev macro-F1 0.6936
[seed 42 | gpu 0] epoch  4  loss 0.3799  clean-dev macro-F1 0.6774
[seed 43 | gpu 1] epoch  4  loss 0.3942  clean-dev macro-F1 0.7016
[seed 42 | gpu 0] epoch  5  loss 0.2401  clean-dev macro-F1 0.6801
[seed 43 | gpu 1] epoch  5  loss 0.2575  clean-dev macro-F1 0.6939
[seed 42 | gpu 0] epoch  6  loss 0.1340  clean-dev macro-F1 0.6851
[seed 43 | gpu 1] epoch  6  loss 0.1586  clean-dev macro-F1 0.6878
[seed 42 | gpu 0] [dev/original/all] macro-F1 0.7058
[seed 42 | gpu 0] [dev/original/clean] macro-F1 0.6858
[seed 42 | gpu 0] [dev/no_tones/all] macro-F1 0.63

## 4 · Results across seeds

Macro-F1 is the primary metric; weighted-F1 is the official SemEval-2023 metric. `clean` excludes tweets that duplicate a training tweet; `no_tones` / `no_diacritics` score the same tweets with tone marks / all diacritics removed.

In [4]:
import pandas as pd

rows = [r for s in SEEDS for r in json.load(open(f"{OUT}/seed{s}/metrics.json"))]
df = pd.DataFrame(rows)
summary = (df.groupby(["split", "subset", "eval_diacritics"])[["macro_f1", "weighted_f1", "accuracy"]]
             .agg(["mean", "std"]).round(4))
summary

macro_f1         weighted_f1         accuracy  \
                                 mean     std        mean     std     mean   
split subset eval_diacritics                                                 
dev   all    no_diacritics     0.6343  0.0161      0.6547  0.0163   0.6584   
             no_tones          0.6374  0.0048      0.6607  0.0012   0.6603   
             original          0.7239  0.0168      0.7442  0.0127   0.7451   
      clean  no_diacritics     0.6173  0.0156      0.6379  0.0162   0.6412   
             no_tones          0.6185  0.0054      0.6419  0.0019   0.6410   
             original          0.6987  0.0118      0.7195  0.0078   0.7204   
test  all    no_diacritics     0.6405  0.0143      0.6572  0.0149   0.6606   
             no_tones          0.6526  0.0037      0.6717  0.0053   0.6696   
             original          0.7281  0.0071      0.7467  0.0055   0.7456   
      clean  no_diacritics     0.6259  0.0131      0.6416  0.0144   0.6449   
             no_tones          0.6355  0.0045      0.6540  0.0063   0.6515   
             original          0.7104  0.0053      0.7291  0.0040   0.7279   

                                      
                                 std  
split subset eval_diacritics          
dev   all    no_diacritics    0.0152  
             no_tones         0.0014  
             original         0.0117  
      clean  no_diacritics    0.0152  
             no_tones         0.0028  
             original         0.0072  
test  all    no_diacritics    0.0130  
             no_tones         0.0070  
             original         0.0058  
      clean  no_diacritics    0.0123  
             no_tones         0.0082  
             original         0.0051

In [5]:
per_seed = df[(df.subset == "clean") & (df.eval_diacritics == "original")][
    ["split", "seed", "macro_f1", "weighted_f1", "f1_negative", "f1_neutral", "f1_positive"]]
per_seed

,split,seed,macro_f1,weighted_f1,f1_negative,f1_neutral,f1_positive
1,dev,42,0.6858,0.7120,0.5485,0.7275,0.7813
7,test,42,0.7059,0.7269,0.5937,0.7321,0.7919
13,dev,43,0.7016,0.7189,0.6173,0.7055,0.7819
19,test,43,0.7092,0.7268,0.6192,0.7167,0.7917
25,dev,44,0.7088,0.7276,0.6118,0.7336,0.7811
31,test,44,0.7162,0.7337,0.6257,0.7271,0.7959
